<a href="https://colab.research.google.com/github/sirrom/Capstone-Project-Intelligent-Healthcare-Diagnostic-Assistant/blob/main/Kmeans_example_unsupervised_learning_lab_work%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sections 1
# What each import does:
# Generate Sample Data
Libraries used:

NumPy handles all mathematical operations and array calculations.

Matplotlib creates all the visualizations and plots.
make_blobs is a tool that generates synthetic datasets with distinct groups of points.

**Data generation:**
The function creates 300 data points arranged into 3 separate clusters. Each cluster has a controlled spread so the groups are visible but not perfectly separated. A random seed is set so the exact same data appears every time the code runs – this ensures all students see identical results.

# The outputs:
X contains the actual coordinates of every point (each point has an x and y value). y_true stores the correct group label for each point – this is the answer key used later to check accuracy.

# The plot:
A scatter plot draws every point on a 2D graph. The x-axis is Feature 1, the y-axis is Feature 2. The title asks students if they can spot the groups visually before the algorithm runs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Generate 3 clusters of points
X, y_true = make_blobs(
    n_samples=300,      # 300 points
    centers=3,          # 3 true clusters
    cluster_std=1.0,    # spread of each cluster
    random_state=42
)

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=50, alpha=0.7)
plt.title("Unlabeled Data – Can you see the groups?")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True, alpha=0.3)
plt.show()

# section 2
This function takes three inputs: the dataset, the number of clusters K, and a maximum number of iterations to prevent infinite looping.

**Step 1** – Random Initialization:
K random data points are selected as the starting centroids. No point can be picked twice. These are the initial "tables" in the cafeteria analogy.

**Step 2** – Main Loop (repeats until convergence):

**Step 2a – Compute Distances:**
For every data point, the straight-line (Euclidean) distance to each centroid is calculated. A grid is created where each row is a point and each column is a centroid, with distances filling the cells.

**Step 2b – Assign Labels:**
Each point is assigned to whichever centroid is closest. The result is a label (0, 1, or 2) for every point. This is the "students sit at the nearest table" step.

**Step 2c – Update Centroids:**
Each centroid moves to the average position of all points assigned to it. If a cluster has no points, the centroid stays where it is to avoid errors.

**Step 2d – Check Convergence:**
If the new centroids are essentially in the same positions as the old ones, the algorithm stops early. Otherwise, centroids are updated and the loop continues.

**Saving history:**
At each iteration, the current centroids and labels are saved into a list. This allows replaying the entire process later as an animation.

# The final output:
The function returns three things – the final centroid positions, the final cluster labels for every point, and the full history for animation.

**The comparison plot:**
Two graphs are shown side by side. The left graph colors points by their true cluster labels. The right graph colors points by what K-Means predicted. The centroids are drawn as large red X's. Students can visually compare how well the algorithm matched reality.

In [ ]:
def kmeans_scratch(X, K, max_iters=10):
    """
    K-Means from scratch – step by step

    Parameters:
    X : array of data points
    K : number of clusters
    max_iters : maximum iterations

    Returns:
    centroids, labels, history (for animation)
    """

    # STEP 1: Randomly initialize K centroids
    # Pick K random points from the data
    random_indices = np.random.choice(len(X), K, replace=False)
    centroids = X[random_indices].copy()

    history = []  # Store centroids at each step for visualization

    for iteration in range(max_iters):
        # STEP 2: Assign each point to the NEAREST centroid
        # Compute Euclidean distance from each point to each centroid
        distances = np.zeros((len(X), K))
        for k in range(K):
            distances[:, k] = np.sqrt(np.sum((X - centroids[k])**2, axis=1))

        # Get the cluster label (0, 1, or 2) for each point
        labels = np.argmin(distances, axis=1)

        # Save state for animation
        history.append((centroids.copy(), labels.copy()))

        # STEP 3: Update centroids – move to center of assigned points
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            # Get all points belonging to cluster k
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                # New centroid = mean of all points in the cluster
                new_centroids[k] = cluster_points.mean(axis=0)
            else:
                # If a cluster gets empty, keep the old centroid
                new_centroids[k] = centroids[k]

        # STEP 4: Check for convergence
        # If centroids didn't move, we're done!
        if np.allclose(centroids, new_centroids):
            print(f"✓ Converged after {iteration + 1} iterations!")
            break

        centroids = new_centroids

    return centroids, labels, history


# Run our K-Means
K = 3
final_centroids, final_labels, history = kmeans_scratch(X, K)

# Visualize the result
plt.figure(figsize=(10, 5))

# Left: True clusters
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title("True Clusters (Ground Truth)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

# Right: K-Means result
plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=final_labels, cmap='viridis', s=50, alpha=0.7)
plt.scatter(final_centroids[:, 0], final_centroids[:, 1],
            c='red', s=200, marker='X', label='Centroids', edgecolors='black')
plt.title(f"K-Means Result (K={K})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.tight_layout()
plt.show()

# Animation of Converging Centroids
This section creates a moving visualization showing how clusters form over time.

A function is set up to redraw the plot for each iteration. At every frame:

**The plot is wiped clean.**

The centroids and labels from that specific iteration are loaded from saved history.

Points are colored by their current cluster assignment.
Centroids are drawn as large red X's.

The title updates to show which iteration number is displayed.

The animation runs with a 700-millisecond delay between frames and loops continuously.
 Students watch as centroids bounce around and then settle into their final positions.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

def animate(iteration):
    ax.clear()
    centroids, labels = history[iteration]

    # Plot points colored by cluster
    for k in range(K):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[k],
                   s=50, alpha=0.6, label=f'Cluster {k}')

    # Plot centroids
    ax.scatter(centroids[:, 0], centroids[:, 1],
               c='red', s=250, marker='X',
               edgecolors='black', linewidth=2, label='Centroids')

    ax.set_title(f'K-Means Iteration {iteration + 1}/{len(history)}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend()
    ax.grid(True, alpha=0.3)

ani = FuncAnimation(fig, animate, frames=len(history), interval=700, repeat=True)
plt.close()
HTML(ani.to_jshtml())  # This creates the interactive animation

# The Elbow Method
**Inertia calculation:**

A function measures how "tight" clusters are by summing the squared distances from every point to its assigned centroid. Lower inertia means points are packed closer together.

**Testing different K values: **

K-Means runs for K values from 1 to 9. For each value, the inertia is calculated and stored.

**The elbow plot:**
A line graph is drawn with K on the x-axis and inertia on the y-axis. As K increases, inertia always decreases (more centroids mean points are closer to one of them).

But at the optimal K, the drop in inertia slows dramatically, forming an "elbow" shape. A vertical red dashed line marks where K=3, the true number of clusters.

The key insight is: adding clusters beyond the elbow gives diminishing returns in tightness.

In [ ]:
def compute_inertia(X, labels, centroids, K):
    inertia = 0
    for k in range(K):
        cluster_points = X[labels == k]
        if len(cluster_points) > 0:
            # Sum of squared distances to centroid
            inertia += np.sum((cluster_points - centroids[k])**2)
    return inertia

# Test different K values
K_range = range(1, 10)
inertias = []

for K in K_range:
    centroids, labels, _ = kmeans_scratch(X, K)
    inertias.append(compute_inertia(X, labels, centroids, K))

# Plot the Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='True K = 3')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method – Find the Best K\n(Look for the "bend" in the curve)')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Real-World Customer Segmentation
# Pandas:
A library that adds spreadsheet-like functionality for organizing and analyzing data.

Simulating customer data:
200 fictitious customers are generated across three hidden segments:


70 budget customers earning around $35,000 with low spending

70 middle-class customers earning around $60,000 with moderate spending

60 premium customers earning around $95,000 with high spending

Each group has realistic variation around its averages.

**Scaling the data:**

A standard scaler transforms the data so all features have equal importance. Without this, Income (ranging from $20k to $120k) would completely dominate Spending Score (ranging from 0 to 100) in distance calculations. After scaling, both features contribute equally.

**Running K-Means properly:**

Scikit-learn's optimized K-Means is used with K=3. It runs 10 times with different random starts and keeps the best result – this solves the initialization sensitivity problem.

**Visualizing results:**

Points are colored by their assigned cluster. Centroids are converted back to original units (actual dollars and spending scores) so axes show meaningful numbers.

**Interpreting segments:**

The code prints the average income and average spending score for each discovered cluster. This helps students give real-world meaning to each group – for example, "Cluster 0 is young budget-conscious shoppers" or "Cluster 2 is affluent high-spenders."

In [ ]:
import pandas as pd

# Simulate customer data
np.random.seed(42)
n_customers = 200

# Generate realistic customer segments
data = {
    'Annual_Income': np.concatenate([
        np.random.normal(35000, 5000, 70),   # Budget customers
        np.random.normal(60000, 7000, 70),    # Middle class
        np.random.normal(95000, 10000, 60)    # Premium customers
    ]),
    'Spending_Score': np.concatenate([
        np.random.normal(30, 10, 70),    # Low spenders
        np.random.normal(60, 10, 70),    # Medium spenders
        np.random.normal(85, 8, 60)      # High spenders
    ])
}

df = pd.DataFrame(data)

# Normalize the data (important for K-Means!)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Apply K-Means
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df['Annual_Income'], df['Spending_Score'],
                      c=df['Cluster'], cmap='Set2', s=100, alpha=0.8)

# Add centroids (in original scale)
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centroids_original[:, 0], centroids_original[:, 1],
            c='red', s=300, marker='X', edgecolors='black', linewidth=2)

plt.xlabel('Annual Income ($)')
plt.ylabel('Spending Score (0-100)')
plt.title('Customer Segmentation using K-Means')
plt.colorbar(scatter, label='Segment')

# Add customer profiles
for i, row in df.iterrows():
    if i % 20 == 0:  # Label every 20th point
        plt.annotate(f'Customer {i}',
                    (row['Annual_Income'], row['Spending_Score']),
                    fontsize=8, alpha=0.7)

plt.grid(True, alpha=0.3)
plt.show()

# Print segment profiles
print("\n📊 Customer Segment Profiles:\n")
for cluster in sorted(df['Cluster'].unique()):
    segment = df[df['Cluster'] == cluster]
    print(f"--- Segment {cluster} ---")
    print(f"  Customers: {len(segment)}")
    print(f"  Avg Income:  ${segment['Annual_Income'].mean():.0f}")
    print(f"  Avg Spending: {segment['Spending_Score'].mean():.1f}/100")
    print()

TEST YOUR KNOWLEDGE

Ensure you execute this

In [6]:
%%capture
pip install fpdf2 PyGithub

In [ ]:
"""
=============================================================================
UNSUPERVISED LEARNING ASSESSMENT SYSTEM
K-MEANS CLUSTERING LABORATORY
=============================================================================
Subject Name    : Artificial Intelligence
Course Level    : 3rd Year Undergraduate
University      : Dedan Kimathi University of Technology
Lab/Module Name : K-Means Clustering Lab
GitHub Repo     : student-ai/KMeans-Lab
GitHub Branch   : main
GitHub Folder   : lab_results
=============================================================================
"""

# === AUTO-INSTALL DEPENDENCIES ===
def _ensure(package, import_name=None):
    import importlib, subprocess, sys
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install",
             package, "--quiet"]
        )

_ensure("fpdf2",    "fpdf")
_ensure("PyGithub", "github")

# === STANDARD LIBRARY IMPORTS ===
import hashlib
import csv
import os
import sys
import time
import json
from datetime import datetime

# === CONFIGURATION BLOCK ===
GITHUB_TOKEN          = "your_github_pat_here"
GITHUB_REPO           = "sirrom/3rd-year-assignment"
GITHUB_BRANCH         = "main"
GITHUB_RESULTS_FOLDER = "lab_kMeans"
RESULTS_CSV           = "AI lab_assessment_results.csv"

# === SECTION NAMES ===
SECTION_NAMES = [
    "Introduction to Unsupervised Learning",
    "K-Means Algorithm Fundamentals",
    "Distance Metrics in Clustering",
    "K-Means Implementation Steps",
    "The Elbow Method",
    "Data Preprocessing for K-Means",
    "K-Means Limitations and Challenges",
    "Real-World Applications of K-Means"
]

# === ANSWER SECURITY SYSTEM ===
_SALT = "DeKUT_NNLab_2024_xQ7#"

def _hash(letter):
    return hashlib.sha256((_SALT + letter).encode()).hexdigest()

def _check(student_answer, stored_hash):
    return hashlib.sha256(
        (_SALT + student_answer).encode()
    ).hexdigest() == stored_hash

def _reveal(stored_hash):
    for L in ("A","B","C","D"):
        if _check(L, stored_hash):
            return L
    return "?"

HA = _hash("A")
HB = _hash("B")
HC = _hash("C")
HD = _hash("D")

# === ALL 28 QUESTIONS ===
QUESTIONS = [
    # ========== SECTION 1: Introduction to Unsupervised Learning (5 Qs) ==========
    {
        "id": "Q1",
        "section": 0,
        "difficulty": "Recall",
        "question": "What distinguishes unsupervised learning from supervised learning?",
        "A": "Unsupervised learning uses labeled data while supervised learning uses unlabeled data",
        "B": "Unsupervised learning finds patterns in unlabeled data without predefined outputs",
        "C": "Unsupervised learning requires a target variable for every data point",
        "D": "Unsupervised learning is only used for regression problems",
        "answer": HB,
        "explain": "Unsupervised learning works with unlabeled data to discover inherent patterns, structures, or groupings. Unlike supervised learning, there is no target variable to predict. The algorithm must find meaningful patterns on its own, such as grouping similar customers together without being told what groups exist."
    },
    {
        "id": "Q2",
        "section": 0,
        "difficulty": "Recall",
        "question": "Which of the following is a primary application of unsupervised learning?",
        "A": "Predicting house prices based on historical data",
        "B": "Classifying emails as spam or not spam",
        "C": "Grouping customers into segments based on purchasing behavior",
        "D": "Predicting stock market prices for next week",
        "answer": HC,
        "explain": "Customer segmentation is a classic unsupervised learning task where customers are grouped based on similarities in their behavior without predefined categories. Options A, B, and D are supervised learning tasks because they require labeled training data with known outcomes."
    },
    {
        "id": "Q3",
        "section": 0,
        "difficulty": "Application",
        "question": "A retail company wants to understand the natural groupings in their customer base to create targeted marketing campaigns. Which approach should they use?",
        "A": "Linear regression to predict customer spending",
        "B": "Logistic regression to classify customer types",
        "C": "Clustering to discover inherent customer segments",
        "D": "Decision trees to predict customer churn",
        "answer": HC,
        "explain": "Clustering is the appropriate unsupervised technique for discovering natural groupings in data without predefined labels. Options A, B, and D are supervised methods that require labeled training data and predefined categories, which do not exist in this scenario."
    },
    {
        "id": "Q4",
        "section": 0,
        "difficulty": "Analysis",
        "question": "Why might unsupervised learning be preferred over supervised learning when exploring a new dataset?",
        "A": "It always produces more accurate results than supervised learning",
        "B": "It requires less computational power and memory",
        "C": "It can reveal hidden patterns and structures that were not previously known",
        "D": "It does not require any data cleaning or preprocessing",
        "answer": HC,
        "explain": "Unsupervised learning excels at exploratory analysis because it can discover unknown patterns, groupings, and relationships in data without being biased by predefined labels. Options A and D are false because unsupervised learning is not inherently more accurate and still requires data preprocessing. Option B is situational and not a guaranteed advantage."
    },
    {
        "id": "Q5",
        "section": 0,
        "difficulty": "Recall",
        "question": "Which of these problems is best suited for unsupervised learning?",
        "A": "Identifying fraudulent credit card transactions",
        "B": "Anomaly detection in network traffic patterns",
        "C": "Predicting tomorrow's weather temperature",
        "D": "Recognizing handwritten digits from images",
        "answer": HB,
        "explain": "Anomaly detection can be framed as an unsupervised problem where the algorithm identifies data points that deviate significantly from normal patterns without requiring labeled examples of anomalies. Options A, C, and D are typically supervised problems that benefit from labeled historical data."
    },

    # ========== SECTION 2: K-Means Algorithm Fundamentals (3 Qs) ==========
    {
        "id": "Q6",
        "section": 1,
        "difficulty": "Recall",
        "question": "What does the letter K represent in the K-Means algorithm?",
        "A": "The number of iterations the algorithm will run",
        "B": "The number of clusters to be formed from the data",
        "C": "The total number of data points in the dataset",
        "D": "The number of features in the dataset",
        "answer": HB,
        "explain": "K represents the number of clusters that the algorithm will partition the data into. This is a user-defined parameter that must be specified before running the algorithm. Options A, C, and D refer to iterations, data points, and features, which are different concepts entirely."
    },
    {
        "id": "Q7",
        "section": 1,
        "difficulty": "Application",
        "question": "In the cafeteria analogy for K-Means, what do the tables represent?",
        "A": "The data points being clustered",
        "B": "The centroids of each cluster",
        "C": "The distance between clusters",
        "D": "The features used for clustering",
        "answer": HB,
        "explain": "In the cafeteria analogy, tables represent centroids. Students (data points) choose the nearest table to sit at, and then tables are moved to the center of the students sitting there. This mirrors how K-Means assigns points to the nearest centroid and then updates centroid positions."
    },
    {
        "id": "Q8",
        "section": 1,
        "difficulty": "Analysis",
        "question": "What happens during the assignment step of K-Means?",
        "A": "Each centroid is moved to the average of all data points",
        "B": "Each data point is assigned to the nearest centroid",
        "C": "New centroids are randomly initialized",
        "D": "The number of clusters K is determined automatically",
        "answer": HB,
        "explain": "During the assignment step, every data point calculates its distance to each centroid and is assigned to the closest one. This creates the cluster groupings. Option A describes the update step, option C describes initialization, and option D is not how K-Means works as K must be specified beforehand."
    },

    # ========== SECTION 3: Distance Metrics in Clustering (3 Qs) ==========
    {
        "id": "Q9",
        "section": 2,
        "difficulty": "Recall",
        "question": "Which distance metric is most commonly used in the standard K-Means algorithm?",
        "A": "Manhattan distance",
        "B": "Euclidean distance",
        "C": "Cosine similarity",
        "D": "Hamming distance",
        "answer": HB,
        "explain": "Euclidean distance (straight-line distance) is the standard distance metric used in K-Means. It calculates the square root of the sum of squared differences between coordinates. Manhattan distance (A) uses absolute differences, cosine similarity (C) measures angles, and Hamming distance (D) compares categorical data."
    },
    {
        "id": "Q10",
        "section": 2,
        "difficulty": "Application",
        "question": "Two data points have coordinates (2, 3) and (5, 7). What is the Euclidean distance between them?",
        "A": "3 units",
        "B": "4 units",
        "C": "5 units",
        "D": "7 units",
        "answer": HC,
        "explain": "The Euclidean distance is calculated as the square root of ((5-2)^2 + (7-3)^2) = sqrt(9 + 16) = sqrt(25) = 5 units. This straight-line distance is what K-Means uses to determine which centroid is closest to each data point."
    },
    {
        "id": "Q11",
        "section": 2,
        "difficulty": "Analysis",
        "question": "Why might Manhattan distance be preferred over Euclidean distance in some clustering applications?",
        "A": "It is always faster to compute for any dataset",
        "B": "It works better with high-dimensional sparse data",
        "C": "It does not require feature scaling",
        "D": "It produces more accurate clusters in all cases",
        "answer": HB,
        "explain": "Manhattan distance can be more effective in high-dimensional spaces where the curse of dimensionality makes Euclidean distance less meaningful. It also tends to be more robust to outliers. However, options A, C, and D are not universally true advantages."
    },

    # ========== SECTION 4: K-Means Implementation Steps (4 Qs) ==========
    {
        "id": "Q12",
        "section": 3,
        "difficulty": "Recall",
        "question": "What is the first step in the K-Means algorithm after choosing K?",
        "A": "Assign each data point to a cluster",
        "B": "Calculate the distance between all pairs of points",
        "C": "Randomly initialize K centroids",
        "D": "Compute the inertia of the clusters",
        "answer": HC,
        "explain": "The first step is to randomly initialize K centroids, typically by selecting K random data points from the dataset as starting positions. Only after initialization can the algorithm proceed to assign points and update centroids."
    },
    {
        "id": "Q13",
        "section": 3,
        "difficulty": "Application",
        "question": "During the update step of K-Means, how is a centroid's new position calculated?",
        "A": "By finding the median of all points in the cluster",
        "B": "By taking the average of all points assigned to that cluster",
        "C": "By selecting the point farthest from the current centroid",
        "D": "By moving the centroid halfway toward each point",
        "answer": HB,
        "explain": "The new centroid position is calculated as the mean (average) of all data points currently assigned to that cluster. This minimizes the within-cluster sum of squares. Options A, C, and D are not how the standard K-Means update step works."
    },
    {
        "id": "Q14",
        "section": 3,
        "difficulty": "Analysis",
        "question": "What does it mean when K-Means has reached convergence?",
        "A": "The algorithm has run for the maximum number of iterations and must stop",
        "B": "All data points have been assigned to unique clusters",
        "C": "The centroids have stopped moving or are changing very little between iterations",
        "D": "The inertia has reached zero and clusters are perfectly separated",
        "answer": HC,
        "explain": "Convergence occurs when centroids stop changing significantly between iterations, indicating the algorithm has found a stable solution. Option A describes hitting the iteration limit, not convergence. Option D is unrealistic as inertia rarely reaches zero in real data."
    },
    {
        "id": "Q15",
        "section": 3,
        "difficulty": "Recall",
        "question": "What is the purpose of the max_iters parameter in K-Means?",
        "A": "To specify the maximum number of clusters the algorithm can create",
        "B": "To limit the number of distance calculations performed",
        "C": "To prevent the algorithm from running indefinitely if it does not converge",
        "D": "To control how many data points are used in each iteration",
        "answer": HC,
        "explain": "The max_iters parameter sets an upper limit on the number of iterations the algorithm will perform. This prevents infinite loops when the algorithm does not converge, ensuring it always finishes within a reasonable time."
    },

    # ========== SECTION 5: The Elbow Method (3 Qs) ==========
    {
        "id": "Q16",
        "section": 4,
        "difficulty": "Recall",
        "question": "What does the Elbow Method help determine in K-Means clustering?",
        "A": "The best distance metric to use",
        "B": "The optimal number of clusters K",
        "C": "The best initialization method for centroids",
        "D": "The maximum number of iterations needed",
        "answer": HB,
        "explain": "The Elbow Method helps determine the optimal value of K by plotting inertia against K values and looking for a bend or elbow in the curve. The point where adding more clusters stops providing significant improvement in tightness is chosen as the optimal K."
    },
    {
        "id": "Q17",
        "section": 4,
        "difficulty": "Application",
        "question": "In the Elbow Method plot, what is shown on the x-axis and y-axis?",
        "A": "X-axis: Inertia, Y-axis: Number of clusters K",
        "B": "X-axis: Number of iterations, Y-axis: Inertia",
        "C": "X-axis: Number of clusters K, Y-axis: Inertia (WCSS)",
        "D": "X-axis: Number of data points, Y-axis: Number of clusters",
        "answer": HC,
        "explain": "The Elbow Method plots the number of clusters K on the x-axis and the inertia (within-cluster sum of squares) on the y-axis. As K increases, inertia decreases, and the optimal K is at the point where the rate of decrease sharply changes, forming an elbow shape."
    },
    {
        "id": "Q18",
        "section": 4,
        "difficulty": "Analysis",
        "question": "Why does inertia always decrease as K increases in the Elbow Method?",
        "A": "Because the algorithm runs faster with more clusters",
        "B": "Because more centroids mean each point is closer to some centroid",
        "C": "Because the distance metric changes with more clusters",
        "D": "Because the data becomes simpler with more clusters",
        "answer": HB,
        "explain": "As K increases, each centroid has fewer points to cover, so points are naturally closer to their assigned centroid, reducing inertia. Adding more clusters always tightens the groupings, but the benefit diminishes after the optimal K, creating the elbow shape."
    },

    # ========== SECTION 6: Data Preprocessing for K-Means (4 Qs) ==========
    {
        "id": "Q19",
        "section": 5,
        "difficulty": "Recall",
        "question": "Why is feature scaling important before applying K-Means?",
        "A": "It reduces the number of data points needed",
        "B": "It ensures all features contribute equally to distance calculations",
        "C": "It eliminates the need to choose a value for K",
        "D": "It guarantees the algorithm will converge faster",
        "answer": HB,
        "explain": "Feature scaling ensures all features have equal weight in distance calculations. Without scaling, features with larger numeric ranges (like income in thousands) would dominate those with smaller ranges (like age 0-100), leading to biased clustering results."
    },
    {
        "id": "Q20",
        "section": 5,
        "difficulty": "Application",
        "question": "A dataset contains Age (0-100) and Annual Income ($20,000-$150,000). What happens if K-Means is applied without scaling?",
        "A": "Both features contribute equally to distance calculations",
        "B": "Age will dominate the clustering because it has a wider range",
        "C": "Income will dominate because its numeric values are much larger",
        "D": "The algorithm will automatically adjust for the difference",
        "answer": HC,
        "explain": "Income values (20,000 to 150,000) are thousands of times larger than Age values (0 to 100), so Income differences will dominate the Euclidean distance calculation. The algorithm effectively ignores Age, producing biased clusters that only reflect income patterns."
    },
    {
        "id": "Q21",
        "section": 5,
        "difficulty": "Recall",
        "question": "Which scaling method transforms data to have a mean of 0 and standard deviation of 1?",
        "A": "Min-Max normalization",
        "B": "Standardization (Z-score normalization)",
        "C": "Robust scaling using median and IQR",
        "D": "Log transformation",
        "answer": HB,
        "explain": "Standardization (Z-score normalization) transforms data by subtracting the mean and dividing by the standard deviation, resulting in a distribution with mean 0 and standard deviation 1. This is commonly used before applying K-Means to ensure fair feature contributions."
    },
    {
        "id": "Q22",
        "section": 5,
        "difficulty": "Analysis",
        "question": "Why is StandardScaler from scikit-learn commonly used before K-Means clustering?",
        "A": "It removes outliers that can distort cluster centers",
        "B": "It makes the data normally distributed, which K-Means requires",
        "C": "It ensures all features have equal influence on the distance metric",
        "D": "It reduces the dimensionality of the dataset",
        "answer": HC,
        "explain": "StandardScaler standardizes features to have similar ranges, ensuring no single feature dominates distance calculations due to its scale. Option A is incorrect because StandardScaler does not remove outliers. Option B is incorrect because K-Means does not require normal distribution."
    },

    # ========== SECTION 7: K-Means Limitations and Challenges (2 Qs) ==========
    {
        "id": "Q23",
        "section": 6,
        "difficulty": "Recall",
        "question": "What shape of clusters does K-Means assume when partitioning data?",
        "A": "Elongated and stretched clusters",
        "B": "Spherical (round) and convex clusters",
        "C": "Interlocking crescent shapes",
        "D": "Clusters of varying densities",
        "answer": HB,
        "explain": "K-Means assumes clusters are spherical and convex because it uses Euclidean distance and assigns each point to the nearest centroid, creating circular decision boundaries. It struggles with elongated shapes, crescents, or non-convex cluster shapes."
    },
    {
        "id": "Q24",
        "section": 6,
        "difficulty": "Analysis",
        "question": "Why does K-Means perform poorly on moon-shaped data?",
        "A": "Because the data has too many features for K-Means to handle",
        "B": "Because moon shapes create non-convex clusters that cross each other",
        "C": "Because the distance calculation is too slow for moon-shaped data",
        "D": "Because moon-shaped data requires more than two clusters",
        "answer": HB,
        "explain": "Moon-shaped data creates interlocking crescent patterns that are not convex or spherical. K-Means tries to separate them with straight-line boundaries, which cuts through the crescents incorrectly. This is a fundamental limitation of the algorithm's spherical cluster assumption."
    },

    # ========== SECTION 8: Real-World Applications of K-Means (4 Qs) ==========
    {
        "id": "Q25",
        "section": 7,
        "difficulty": "Application",
        "question": "A marketing team segments customers using K-Means on income and spending score. What business value does this provide?",
        "A": "It predicts exactly how much each customer will spend",
        "B": "It identifies natural customer groups for targeted marketing campaigns",
        "C": "It replaces the need for any market research",
        "D": "It automatically generates personalized product recommendations",
        "answer": HB,
        "explain": "Customer segmentation using K-Means reveals natural groupings in customer behavior, allowing marketers to create targeted campaigns for each segment. Options A and D overstate what K-Means provides, as it only groups customers without making predictions or recommendations."
    },
    {
        "id": "Q26",
        "section": 7,
        "difficulty": "Recall",
        "question": "In image compression, how is K-Means clustering applied?",
        "A": "By identifying objects in an image",
        "B": "By reducing the number of colors in the image to K representative colors",
        "C": "By detecting edges and boundaries in images",
        "D": "By splitting the image into K equal-sized regions",
        "answer": HB,
        "explain": "K-Means can compress images by clustering all pixels into K color groups, then replacing each pixel with its cluster's centroid color. This reduces the color palette to K colors while preserving the overall image appearance."
    },
    {
        "id": "Q27",
        "section": 7,
        "difficulty": "Application",
        "question": "A hospital wants to group patients based on medical test results to identify different health profiles. Which approach is most appropriate?",
        "A": "Use supervised classification with labeled patient diagnoses",
        "B": "Apply K-Means clustering to discover natural patient groupings",
        "C": "Use linear regression to predict health outcomes",
        "D": "Apply a decision tree to classify disease types",
        "answer": HB,
        "explain": "K-Means clustering is appropriate because the goal is to discover natural groupings in patient data without predefined health profiles. Options A, C, and D are supervised methods that require labeled training data, which may not exist for discovering new patterns."
    },
    {
        "id": "Q28",
        "section": 7,
        "difficulty": "Analysis",
        "question": "A company applies K-Means to their customer data with K=5 and gets poor results. What is the most likely first troubleshooting step?",
        "A": "Increase K to 10 to create more detailed segments",
        "B": "Check whether feature scaling was applied before clustering",
        "C": "Switch to a supervised learning algorithm immediately",
        "D": "Reduce the number of features to only one",
        "answer": HB,
        "explain": "Feature scaling is one of the most common and impactful preprocessing steps for K-Means. If features have different scales, the results can be heavily biased. Checking scaling should be the first troubleshooting step before trying different K values or changing algorithms."
    }
]

# === DATETIME FOR TIMESTAMPS ===
def get_timestamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def get_date():
    return datetime.now().strftime("%Y-%m-%d")

# === PDF SAFE STRING SANITIZER ===
def pdf_safe(text):
    return str(text).encode("latin-1", errors="ignore").decode("latin-1")

# === HELPER: BOXED PRINT ===
def box_print(lines, width=60):
    print("=" * width)
    for line in lines:
        print(f"  {line}")
    print("=" * width)

# === WRAP TEXT FOR DISPLAY ===
def wrap_text(text, width=58):
    words = text.split()
    lines = []
    current = ""
    for word in words:
        if len(current) + len(word) + 1 <= width:
            current = current + " " + word if current else word
        else:
            lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines

# === STUDENT REGISTRATION ===
def register_student():
    while True:
        print("\n" + "=" * 60)
        print("  DEDAN KIMATHI UNIVERSITY OF TECHNOLOGY")
        print("  Department of Computer Science")
        print("  Artificial Intelligence - K-Means Clustering Lab")
        print("=" * 60)

        reg_no = input("\nEnter Registration Number: ").strip()
        while len(reg_no) < 5:
            print("  Registration number must be at least 5 characters.")
            reg_no = input("Enter Registration Number: ").strip()

        full_name = input("\nEnter Full Name (Surname First): ").strip()
        while len(full_name.split()) < 2:
            print("  Please enter both first and last name.")
            full_name = input("Enter Full Name (Surname First): ").strip()

        print("\n" + "-" * 40)
        print("  PLEASE CONFIRM YOUR DETAILS")
        print("-" * 40)
        print(f"  Registration No: {reg_no}")
        print(f"  Full Name:       {full_name}")
        print("-" * 40)

        confirm = input("\nConfirm details correct? (Y/N): ").strip().upper()
        if confirm == "Y":
            safe_reg = reg_no.replace("/", "_").replace(" ", "_").replace("\\", "_")
            timestamp = get_timestamp()
            date = get_date()
            csv_file = f"result_{safe_reg}.csv"
            pdf_file = f"result_{safe_reg}.pdf"

            print("\n  Starting assessment in 3 seconds...")
            for i in range(3, 0, -1):
                print(f"  {i}...")
                time.sleep(1)
            print("  Begin!\n")

            return {
                "reg_no": reg_no,
                "full_name": full_name,
                "timestamp": timestamp,
                "date": date,
                "safe_reg": safe_reg,
                "csv_file": csv_file,
                "pdf_file": pdf_file
            }
        else:
            print("\n  Restarting registration...\n")

# === RUN ASSESSMENT ===
def run_assessment(student, demo_mode=False):
    section_results = {i: {"correct": 0, "total": 0} for i in range(8)}
    wrong_questions = []
    score = 0
    total = len(QUESTIONS)

    for idx, q in enumerate(QUESTIONS):
        section_idx = q["section"]
        section_results[section_idx]["total"] += 1

        print("\n" + "-" * 60)
        print(f"  Question {idx+1} of {total} | {SECTION_NAMES[section_idx]}")
        print("-" * 60)

        wrapped = wrap_text(q["question"])
        for line in wrapped:
            print(f"  {line}")
        print()

        print(f"  A) {q['A']}")
        print(f"  B) {q['B']}")
        print(f"  C) {q['C']}")
        print(f"  D) {q['D']}")
        print()

        if demo_mode:
            answer = _reveal(q["answer"])
            print(f"  [DEMO] Auto-answering: {answer}")
        else:
            answer = input("  Your answer (A/B/C/D): ").strip().upper()
            while answer not in ("A", "B", "C", "D"):
                answer = input("  Invalid. Please enter A, B, C, or D: ").strip().upper()

        if _check(answer, q["answer"]):
            print("  [CORRECT]")
            score += 1
            section_results[section_idx]["correct"] += 1
        else:
            correct_letter = _reveal(q["answer"])
            print(f"  [WRONG] Correct answer was: {correct_letter}")
            wrong_questions.append(q["id"])

        print("\n  Explanation:")
        wrapped_exp = wrap_text(q["explain"], width=56)
        for line in wrapped_exp:
            print(f"    {line}")

    return score, total, wrong_questions, section_results

# === GENERATE PDF REPORT ===
def generate_pdf(student, score, total, percentage, grade,
                  wrong_questions, section_results):
    from fpdf import FPDF

    pdf = FPDF()
    pdf.add_page()

    # === HEADER BAR ===
    pdf.set_fill_color(26, 60, 120)
    pdf.rect(0, 0, 210, 25, 'F')
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 16)
    pdf.set_xy(10, 5)
    pdf.cell(190, 10, pdf_safe("DEDAN KIMATHI UNIVERSITY OF TECHNOLOGY"), 0, 1, "C")
    pdf.set_font("Helvetica", "", 10)
    pdf.set_xy(10, 16)
    pdf.cell(190, 5, pdf_safe("Artificial Intelligence - K-Means Clustering Lab"), 0, 1, "C")
    pdf.set_xy(10, 21)
    pdf.cell(190, 4, pdf_safe(f"Generated: {get_timestamp()}"), 0, 1, "C")

    pdf.set_text_color(0, 0, 0)
    pdf.ln(15)

    # === STUDENT INFO TABLE ===
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(190, 7, pdf_safe("STUDENT INFORMATION"), 0, 1, "L")
    pdf.set_font("Helvetica", "", 10)

    info_lines = [
        ("Name:", student["full_name"]),
        ("Registration No:", student["reg_no"]),
        ("Submitted:", student["timestamp"])
    ]

    for label, value in info_lines:
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(40, 6, pdf_safe(label), 0, 0)
        pdf.set_font("Helvetica", "", 10)
        pdf.cell(0, 6, pdf_safe(value), 0, 1)

    pdf.ln(5)

    # === SCORE BANNER ===
    if percentage >= 75:
        bg_color = (34, 139, 34)  # green
    elif percentage >= 60:
        bg_color = (255, 165, 0)  # orange
    else:
        bg_color = (220, 20, 60)  # red

    pdf.set_fill_color(*bg_color)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 20)
    pdf.cell(190, 18, pdf_safe(f"SCORE: {score}/{total} = {percentage:.1f}%  Grade: {grade}"), 0, 1, "C", True)
    pdf.set_text_color(0, 0, 0)
    pdf.ln(5)

    # === SECTION BREAKDOWN ===
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(190, 7, pdf_safe("SECTION BREAKDOWN"), 0, 1, "L")

    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(80, 5, pdf_safe("Section"), 1, 0)
    pdf.cell(40, 5, pdf_safe("Score"), 1, 0, "C")
    pdf.cell(40, 5, pdf_safe("Percentage"), 1, 0, "C")
    pdf.cell(30, 5, pdf_safe("Status"), 1, 1, "C")

    pdf.set_font("Helvetica", "", 9)
    for i in range(8):
        c = section_results[i]["correct"]
        t = section_results[i]["total"]
        pct = (c / t * 100) if t > 0 else 0

        if pct >= 75:
            status = "PASS"
        elif pct >= 50:
            status = "OK"
        else:
            status = "FAIL"

        name = SECTION_NAMES[i][:40]
        pdf.cell(80, 5, pdf_safe(name), 1, 0)
        pdf.cell(40, 5, pdf_safe(f"{c}/{t}"), 1, 0, "C")
        pdf.cell(40, 5, pdf_safe(f"{pct:.1f}%"), 1, 0, "C")
        pdf.cell(30, 5, pdf_safe(status), 1, 1, "C")

    pdf.ln(5)

    # === QUESTIONS TO REVIEW ===
    if wrong_questions:
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(190, 7, pdf_safe("QUESTIONS TO REVIEW"), 0, 1, "L")
        pdf.ln(2)

        for q in QUESTIONS:
            if q["id"] in wrong_questions:
                pdf.set_font("Helvetica", "B", 10)
                pdf.cell(190, 5, pdf_safe(f"{q['id']}: {q['question'][:70]}..."), 0, 1)
                pdf.set_font("Helvetica", "I", 9)
                wrapped_exp = wrap_text(q["explain"], width=90)
                for line in wrapped_exp:
                    pdf.cell(190, 4, pdf_safe(line), 0, 1)
                pdf.ln(3)

    # === FOOTER ===
    pdf.set_y(-30)
    pdf.set_font("Helvetica", "I", 8)
    pdf.cell(190, 5, pdf_safe("Dedan Kimathi University of Technology"), 0, 1, "C")
    pdf.cell(190, 5, pdf_safe(f"GitHub: {GITHUB_REPO}"), 0, 1, "C")

    pdf.output(student["pdf_file"])
    print(f"\n  PDF saved: {student['pdf_file']}")

# === SAVE CSV ===
def save_csv(student, score, total, percentage, grade, wrong_questions,
             section_results, master_mode=False):
    # Per-student CSV
    wrong_str = "|".join(wrong_questions) if wrong_questions else "None"

    row = [
        student["timestamp"],
        student["reg_no"],
        student["full_name"],
        str(score),
        str(total),
        f"{percentage:.1f}",
        grade,
        wrong_str
    ]

    for i in range(8):
        c = section_results[i]["correct"]
        t = section_results[i]["total"]
        row.append(f"{c}/{t}")

    # Write per-student CSV
    with open(student["csv_file"], "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "reg_no", "full_name", "score",
                         "total", "percentage", "grade", "wrong_questions"]
                         + SECTION_NAMES)
        writer.writerow(row)
    print(f"  CSV saved: {student['csv_file']}")

    # Append to master CSV
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["timestamp", "reg_no", "full_name", "score",
                           "total", "percentage", "grade", "wrong_questions"]
                           + SECTION_NAMES)
        writer.writerow(row)
    print(f"  Appended to: {RESULTS_CSV}")

# === GITHUB UPLOAD ===
def upload_to_github(student):
    if GITHUB_TOKEN == "your_github_pat_here":
        print("\n  GITHUB_TOKEN not configured. Skipping GitHub upload.")
        return

    try:
        from github import Github, GithubException

        g = Github(GITHUB_TOKEN)
        repo = g.get_repo(GITHUB_REPO)

        files_to_upload = [
            student["csv_file"],
            RESULTS_CSV,
            student["pdf_file"]
        ]

        for filename in files_to_upload:
            folder_path = f"{GITHUB_RESULTS_FOLDER}/{filename}"

            try:
                contents = repo.get_contents(folder_path, ref=GITHUB_BRANCH)
                sha = contents.sha

                with open(filename, "r") if filename.endswith((".csv", ".txt")) else open(filename, "rb") as f:
                    content = f.read()

                if filename.endswith(".pdf"):
                    repo.update_file(folder_path, f"Update {filename}",
                                   content, sha, branch=GITHUB_BRANCH)
                else:
                    repo.update_file(folder_path, f"Update {filename}",
                                   content.decode(), sha, branch=GITHUB_BRANCH)
                print(f"  Updated on GitHub: {folder_path}")

            except GithubException as e:
                if e.status == 404:
                    with open(filename, "r") if filename.endswith((".csv", ".txt")) else open(filename, "rb") as f:
                        content = f.read()

                    if filename.endswith(".pdf"):
                        repo.create_file(folder_path, f"Create {filename}",
                                       content, branch=GITHUB_BRANCH)
                    else:
                        repo.create_file(folder_path, f"Create {filename}",
                                       content.decode(), branch=GITHUB_BRANCH)
                    print(f"  Created on GitHub: {folder_path}")
                else:
                    print(f"  GitHub error for {filename}: {e}")

    except Exception as e:
        print(f"\n  GitHub upload error: {e}")

# === MODE: demo ===
def run_demo():
    print("\n" + "=" * 60)
    print("  DEMO MODE: Auto-answering all questions correctly")
    print("=" * 60)

    student = {
        "reg_no": "DEMO/001/2024",
        "full_name": "Demo Student",
        "timestamp": get_timestamp(),
        "date": get_date(),
        "safe_reg": "DEMO_001_2024",
        "csv_file": "result_DEMO_001_2024.csv",
        "pdf_file": "result_DEMO_001_2024.pdf"
    }

    score, total, wrong_questions, section_results = run_assessment(student, demo_mode=True)
    display_results(student, score, total, wrong_questions, section_results)

# === MODE: results ===
def show_results():
    if not os.path.isfile(RESULTS_CSV):
        print("\n  No results file found.")
        return

    print("\n" + "=" * 60)
    print("  ALL SUBMITTED RESULTS")
    print("=" * 60)

    with open(RESULTS_CSV, "r") as f:
        reader = csv.reader(f)
        rows = list(reader)

    if len(rows) <= 1:
        print("  No submissions yet.")
        return

    headers = rows[0]
    print(f"\n  {'Reg No':<20} {'Name':<25} {'Score':<10} {'Grade':<5}")
    print("  " + "-" * 60)

    for row in rows[1:]:
        reg = row[1][:20]
        name = row[2][:25]
        score_txt = row[3] + "/" + row[4]
        grade = row[6]
        print(f"  {reg:<20} {name:<25} {score_txt:<10} {grade:<5}")

    print(f"\n  Total submissions: {len(rows) - 1}")

# === MODE: key ===
def show_answer_key():
    print("\n" + "=" * 60)
    print("  ANSWER KEY")
    print("=" * 60)

    for q in QUESTIONS:
        correct = _reveal(q["answer"])
        print(f"\n  {q['id']}: {correct}")
        print(f"    {q['question'][:70]}")

# === DISPLAY RESULTS ===
def display_results(student, score, total, wrong_questions, section_results):
    percentage = (score / total) * 100

    if percentage >= 90:
        grade = "A"
        remark = "Excellent!"
    elif percentage >= 75:
        grade = "B"
        remark = "Very Good!"
    elif percentage >= 60:
        grade = "C"
        remark = "Good"
    elif percentage >= 50:
        grade = "D"
        remark = "Satisfactory"
    else:
        grade = "F"
        remark = "Needs Improvement"

    print("\n" + "=" * 60)
    print("  YOUR RESULTS")
    print("=" * 60)
    print(f"\n  Name:              {student['full_name']}")
    print(f"  Registration No:   {student['reg_no']}")
    print(f"  Submitted:         {student['timestamp']}")
    print(f"\n  Score:             {score}/{total} ({percentage:.1f}%)")
    print(f"  Grade:             {grade} - {remark}")

    print("\n" + "-" * 60)
    print("  SECTION BREAKDOWN")
    print("-" * 60)

    for i in range(8):
        c = section_results[i]["correct"]
        t = section_results[i]["total"]
        pct = (c / t * 100) if t > 0 else 0

        if pct >= 75:
            status = "PASS"
        elif pct >= 50:
            status = "OK"
        else:
            status = "FAIL"

        bar_length = 20
        filled = int(pct / 100 * bar_length)
        bar = "#" * filled + "." * (bar_length - filled)

        print(f"  [{status}] {SECTION_NAMES[i][:35]:<35} {c}/{t:<3} [{bar}] {pct:.0f}%")

    if wrong_questions:
        print("\n  WRONG QUESTIONS:")
        for q in QUESTIONS:
            if q["id"] in wrong_questions:
                correct = _reveal(q["answer"])
                print(f"    {q['id']}: Correct answer was {correct}")
    else:
        print("\n  PERFECT SCORE! No wrong answers.")

    return percentage, grade

# === MAIN ENTRY POINT ===
if __name__ == "__main__":
    # Filter out Jupyter arguments
    user_args = [
        a for a in sys.argv[1:]
        if not a.startswith("-") and not a.endswith(".json")
    ]

    if "demo" in user_args:
        run_demo()

    elif "results" in user_args:
        show_results()

    elif "key" in user_args:
        show_answer_key()

    else:
        # Normal student assessment flow
        student = register_student()
        score, total, wrong_questions, section_results = run_assessment(student)

        percentage, grade = display_results(
            student, score, total, wrong_questions, section_results
        )

        print("\n" + "-" * 60)
        print("  SAVING RESULTS...")
        print("-" * 60)

        save_csv(student, score, total, percentage, grade,
                 wrong_questions, section_results)

        generate_pdf(student, score, total, percentage, grade,
                     wrong_questions, section_results)

        upload_to_github(student)

        print("\n" + "=" * 60)
        print("  ASSESSMENT COMPLETE. Thank you!")
        print("=" * 60)